In [0]:
from pyspark.sql.types import StructType, StructField, StringType
from pyspark.sql.functions import col, current_timestamp
from pyspark.sql.functions import to_date, date_format, coalesce, lit, concat, lpad
import re

In [0]:
# Autenticación hacia adlssmartdatamanuel95 usando Account Key desde Databricks Secrets
storage_account = "adlssmartdatamanuel95"

spark.conf.set(
    f"fs.azure.account.key.{storage_account}.dfs.core.windows.net",
    dbutils.secrets.get(scope="accessScopeforADLS", key="storageAccessKey")
)

print(f"✅ Autenticación configurada para: {storage_account}")

In [0]:
# Configuración de rutas
storage_account = "adlssmartdatamanuel95"
container_origen = "raw"           # contenedor donde subes el CSV en Azure

ruta_origen   = f"abfss://{container_origen}@{storage_account}.dfs.core.windows.net/"
tabla_destino = "adbsmartdatamanuelestrada.bronze.base_produccion_brz"

print("="*60)
print("CONFIGURACIÓN DEL PIPELINE")
print("="*60)
print(f"Ruta origen: {ruta_origen}")
print(f"Tabla destino: {tabla_destino}")
print("="*60)

In [0]:
print("="*60)
print("LECTURA DEL ARCHIVO CSV")
print("="*60)

# ============================================================
# PASO 1: Extraer mes del nombre del archivo
# ============================================================
archivos = dbutils.fs.ls(ruta_origen)
# Filtrar SOLO archivos que empiezan con "base_produccion" y terminan en .csv
archivos_csv = [f.name for f in archivos if f.name.startswith('base_produccion') and f.name.endswith('.csv')]

if len(archivos_csv) == 0:
    raise ValueError(f"❌ ERROR: No se encontró ningún archivo 'base_produccion*.csv' en {ruta_origen}")

nombre_archivo = archivos_csv[0]
print(f"\nArchivo detectado: {nombre_archivo}")

# Extraer YYYYMM del nombre (ej: base_produccion_202601.csv -> 202601)
match = re.search(r'(\d{6})', nombre_archivo)
if not match:
    raise ValueError(f"❌ ERROR: No se pudo extraer el mes del nombre del archivo: {nombre_archivo}")

mes_archivo = match.group(1)
ano_archivo = mes_archivo[:4]
mes_num_archivo = mes_archivo[4:6]
print(f"Mes del archivo: {mes_archivo} ({mes_num_archivo}/{ano_archivo})")

# ============================================================
# VALIDACIÓN #1: Verificar si el mes ya existe en la tabla
# ============================================================
print("\n" + "-"*60)
print("VALIDACIÓN #1: Control de Duplicados por Mes")
print("-"*60)

try:
    tabla_existente = spark.table(tabla_destino)
    
    # Extraer meses existentes en la tabla
    meses_existentes = tabla_existente.withColumn(
        "fecha_parsed_tabla",
        to_date(col("fecha_orden"), "d/M/yyyy")
    ).select(
        coalesce(
            concat(
                date_format(col("fecha_parsed_tabla"), "yyyy"),
                lpad(date_format(col("fecha_parsed_tabla"), "MM"), 2, "0")
            ),
            lit("UNKNOWN")
        ).alias("mes_tabla")
    ).distinct().collect()
    
    meses_en_tabla = [row['mes_tabla'] for row in meses_existentes]
    print(f"Meses existentes en tabla: {', '.join(meses_en_tabla)}")
    
    if mes_archivo in meses_en_tabla:
        raise ValueError(
            f"❌ ERROR: El mes {mes_archivo} ya existe en la tabla Bronze.\n"
            f"   No se puede cargar el mismo mes dos veces.\n"
            f"   Para recargar, primero elimine los datos del mes: \n"
            f"   DELETE FROM {tabla_destino} WHERE to_date(fecha_orden, 'd/M/yyyy') \n"
            f"   BETWEEN '{ano_archivo}-{mes_num_archivo}-01' AND '{ano_archivo}-{mes_num_archivo}-31'"
        )
    
    print(f"✅ El mes {mes_archivo} NO existe en la tabla. Se procederá con la carga.")
    
except Exception as e:
    if "Table or view not found" in str(e) or "cannot be found" in str(e).lower():
        print(f"ℹ️  La tabla no existe. Se creará con los datos del mes {mes_archivo}.")
    else:
        raise

# ============================================================
# PASO 2: Leer el CSV
# ============================================================
print("\n" + "-"*60)
print("Lectura del archivo CSV")
print("-"*60)

# Construir ruta completa al archivo específico
ruta_completa = ruta_origen + nombre_archivo
print(f"Leyendo: {ruta_completa}")

df = (spark.read
  .format("csv")
  .option("header", "true")
  .option("delimiter", ";")
  .option("inferSchema", "false")
  .load(ruta_completa)
)

print(f"✅ CSV cargado: {len(df.columns)} columnas")
print(f"Columnas: {', '.join(df.columns)}")

# ============================================================
# VALIDACIÓN #2: Verificar que el DataFrame NO esté vacío
# ============================================================
print("\n" + "-"*60)
print("VALIDACIÓN #2: DataFrame Vacío")
print("-"*60)

total_registros = df.count()
print(f"Registros en el archivo: {total_registros:,}")

if total_registros == 0:
    raise ValueError(
        f"❌ ERROR: El archivo {nombre_archivo} está vacío.\n"
        f"   No se puede cargar un archivo sin registros."
    )

print(f"✅ El archivo contiene {total_registros:,} registros.")
print("\n" + "="*60)

In [0]:
print("="*60)
print("VALIDACIONES DE CONTENIDO")
print("="*60)

# ============================================================
# VALIDACIÓN #3: Un solo mes en el archivo
# ============================================================
print("\nVALIDACIÓN #3: Un Solo Mes en el Archivo")
print("-"*60)

# Parsear fechas y extraer mes
df_with_month = df.withColumn(
    "fecha_parsed",
    to_date(col("Fecha Orden"), "d/M/yyyy")
).withColumn(
    "mes_registro",
    coalesce(
        concat(
            date_format(col("fecha_parsed"), "yyyy"),
            lpad(date_format(col("fecha_parsed"), "MM"), 2, "0")
        ),
        lit(mes_archivo)
    )
)

# Contar registros por mes
mes_counts = df_with_month.groupBy("mes_registro").count().collect()
print(f"\nDistribución de registros por mes:")
for row in mes_counts:
    print(f"  Mes {row['mes_registro']}: {row['count']:,} registros")

if len(mes_counts) > 1:
    meses_encontrados = [row['mes_registro'] for row in mes_counts]
    raise ValueError(
        f"❌ ERROR: El archivo contiene registros de múltiples meses.\n"
        f"   Mes del archivo: {mes_archivo}\n"
        f"   Meses encontrados: {', '.join(meses_encontrados)}\n"
        f"   El archivo debe contener datos de UN solo mes."
    )

print(f"✅ El archivo contiene UN solo mes")

# ============================================================
# VALIDACIÓN #4: El mes coincide con el nombre del archivo
# ============================================================
print("\nVALIDACIÓN #4: Coherencia Mes del Archivo vs Contenido")
print("-"*60)

mes_registros = mes_counts[0]['mes_registro']
print(f"Mes del archivo: {mes_archivo}")
print(f"Mes de los registros: {mes_registros}")

if mes_registros != mes_archivo:
    raise ValueError(
        f"❌ ERROR: El contenido NO coincide con el nombre del archivo.\n"
        f"   Mes del archivo: {mes_archivo}\n"
        f"   Mes de los registros: {mes_registros}\n"
        f"   El nombre del archivo y el contenido deben coincidir."
    )

print(f"✅ Coherencia validada: Mes del archivo = Mes de registros")

print("\n" + "="*60)
print("✅ TODAS LAS VALIDACIONES APROBADAS")
print("="*60)

In [0]:
print("="*60)
print("ESTANDARIZACIÓN DE COLUMNAS")
print("="*60)

# Renombrar columnas y agregar timestamp
df_renamed = df_with_month \
  .withColumnRenamed("Fecha Orden", "fecha_orden") \
  .withColumnRenamed("CodVenta", "cod_venta") \
  .withColumnRenamed("CodTienda", "cod_tienda") \
  .withColumnRenamed("CodProducto", "cod_producto") \
  .withColumnRenamed("CodCliente", "cod_cliente") \
  .withColumnRenamed("Cantidad", "cantidad") \
  .withColumnRenamed("Precio", "precio_unit") \
  .withColumnRenamed("Coste", "coste_unit") \
  .withColumnRenamed("%IVA", "porc_iva") \
  .withColumn("data_ingestion_ts", current_timestamp().cast("string"))

print(f"\nColumnas estandarizadas:")
for i, col_name in enumerate(df_renamed.columns, 1):
    if col_name not in ["fecha_parsed", "mes_registro"]:
        print(f"  {i}. {col_name}")

print("\n✅ Estandarización completa")
print("="*60)

In [0]:
print("="*60)
print("ESCRITURA A TABLA BRONZE")
print(f"Destino: {tabla_destino}")
print("="*60)

# Remover columnas temporales antes de escribir
df_final = df_renamed.select([c for c in df_renamed.columns if c not in ["mes_registro", "fecha_parsed"]])

print(f"\nRegistros a insertar: {total_registros:,}")
print(f"Mes: {mes_archivo}")

df_final.write \
  .format("delta") \
  .mode("append") \
  .saveAsTable(tabla_destino)

print("\n" + "="*60)
print("✅ INSERCIÓN EXITOSA")
print("="*60)
print(f"Tabla: {tabla_destino}")
print(f"Mes insertado: {mes_archivo}")
print(f"Registros insertados: {total_registros:,}")
print(f"Total registros en tabla: {spark.table(tabla_destino).count():,}")
print("="*60)
